# Day 017 | NumPy — Linear Algebra (SVD, Eigenvalues)

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** 2026-05-16  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| Vectors & matrices | Structured numerical arrays | Stress vectors, stiffness matrices |
| Dot product / `@` | Matrix multiplication | Coordinate transformations |
| `np.linalg.solve` | Solve Ax = b | Find unknowns in a settlement problem |
| Eigenvalues & eigenvectors | Principal directions & magnitudes | Principal stresses from stress tensor |
| Mohr's circle | 2D stress visualization | Max shear and principal stresses |
| SVD | Decompose a matrix into components | Dimensionality reduction preview |
| Determinant & inverse | Matrix properties | Check if stiffness matrix is singular |

---

## Our Geotechnical Problem

A soil element at depth is subjected to **combined loading** — vertical stress, horizontal stress, and shear stress. The **stress tensor** is a 2×2 (or 3×3) matrix:

```
σ = | σxx  τxy |
    | τxy  σyy |
```

To find the **principal stresses** (σ1, σ3) — the maximum and minimum normal stresses on any plane — we need **eigenvalues** of this tensor. The **eigenvectors** give us the orientation of those planes.

This is the **exact same math** behind PCA (Principal Component Analysis) in ML — finding the directions of maximum variance in data.

---

## Part 1: Why Linear Algebra for Geotechnical Engineering

Linear algebra appears everywhere in geotechnical engineering:

| Application | Matrix/Vector Involved |
|-------------|------------------------|
| Stress analysis | Stress tensor (2×2 or 3×3) |
| Finite element method | Stiffness matrix **K**, force vector **F** |
| Settlement of layered soil | System of equations for each layer |
| Slope stability | Equilibrium equations for slices |
| Coordinate transforms | Rotation matrices for inclined planes |

And in ML, the same tools appear as:
- **PCA** = eigenvalue decomposition of the covariance matrix
- **SVD** = the engine behind dimensionality reduction
- **Linear regression** = solving a system of normal equations

Today we bridge these two worlds.

---

## Part 2: Vectors & Matrices in NumPy

In geotechnical engineering:
- A **vector** represents forces, displacements, or stress components at a point
- A **matrix** represents a stress tensor, stiffness matrix, or transformation

NumPy handles both as arrays of different dimensions.

In [ ]:
# ─────────────────────────────────────────────────────
# VECTORS & MATRICES — Stress and stiffness in NumPy
# ─────────────────────────────────────────────────────

import numpy as np

# ─── Vectors ───
# Force vector at a foundation node (Fx, Fy, Fz) in kN
force = np.array([0, -500, 0])  # 500 kN downward
print(f"Force vector:  {force}  shape: {force.shape}")

# Displacement vector (dx, dy, dz) in mm
displacement = np.array([2.1, -12.5, 0.3])
print(f"Displacement:  {displacement}  shape: {displacement.shape}")

# Stress vector (σxx, σyy, τxy) in kPa
stress_vec = np.array([120, 80, 25])
print(f"Stress vector: {stress_vec}  shape: {stress_vec.shape}")
print()

# ─── Matrices ───
# 2D stress tensor
stress_tensor = np.array([
    [120, 25],   # [σxx, τxy]
    [ 25, 80],   # [τxy, σyy]
])
print(f"2D Stress Tensor (kPa):")
print(stress_tensor)
print(f"  shape: {stress_tensor.shape}, symmetric: {np.allclose(stress_tensor, stress_tensor.T)}")
print()

# 3D stress tensor
stress_3d = np.array([
    [120,  25,  10],   # [σxx, τxy, τxz]
    [ 25,  80,  15],   # [τyx, σyy, τyz]
    [ 10,  15, 100],   # [τzx, τzy, σzz]
])
print(f"3D Stress Tensor (kPa):")
print(stress_3d)
print(f"  shape: {stress_3d.shape}, symmetric: {np.allclose(stress_3d, stress_3d.T)}")

---

## Part 3: Matrix Operations — Dot Product, Multiply, Transpose

| Operation | NumPy Syntax | Meaning |
|-----------|-------------|--------|
| Dot product | `np.dot(a, b)` or `a @ b` | Scalar product of vectors, matrix multiply |
| Matrix multiply | `A @ B` | Standard matrix multiplication |
| Transpose | `A.T` | Swap rows and columns |
| Element-wise multiply | `A * B` | Multiply corresponding elements (NOT matrix multiply) |

In [ ]:
# ─────────────────────────────────────────────────────
# MATRIX OPERATIONS — Geotechnical applications
# ─────────────────────────────────────────────────────

import numpy as np

# ─── Dot product: stress × area = force ───
# Traction on a plane with unit normal n = [cos(30°), sin(30°)]
stress_tensor = np.array([
    [120, 25],
    [ 25, 80],
])
theta = np.radians(30)
normal = np.array([np.cos(theta), np.sin(theta)])

# Traction vector: t = σ · n
traction = stress_tensor @ normal
print(f"Normal vector (30°):  {np.round(normal, 4)}")
print(f"Traction vector:      {np.round(traction, 2)} kPa")
print()

# ─── Coordinate transformation ───
# Rotate stress tensor by angle θ: σ' = R σ R^T
R = np.array([
    [ np.cos(theta), np.sin(theta)],
    [-np.sin(theta), np.cos(theta)],
])

stress_rotated = R @ stress_tensor @ R.T
print(f"Rotation matrix R (30°):")
print(np.round(R, 4))
print()
print(f"Original stress tensor:")
print(stress_tensor)
print(f"\nRotated stress tensor (30°):")
print(np.round(stress_rotated, 2))
print()

# ─── Work done: W = F · d (dot product of vectors) ───
force = np.array([0, -500, 0])  # kN
disp = np.array([0.002, -0.0125, 0.0003])  # meters
work = np.dot(force, disp)
print(f"Work done = F · d = {work:.4f} kN·m = {work*1000:.2f} J")

---

## Part 4: Solving Linear Systems — Settlement of a 3-Layer Soil

Many geotechnical problems reduce to solving **Ax = b**:

- **A** = coefficient matrix (material properties, geometry)
- **x** = unknowns (settlements, forces, pressures)
- **b** = known values (applied loads, boundary conditions)

**Example:** A foundation sits on 3 soil layers. Each layer compresses under load, but adjacent layers share stress. The system of equations is:

```
E1/H1 · s1  - E1/H1 · s2               = q
-E1/H1 · s1 + (E1/H1 + E2/H2) · s2 - E2/H2 · s3 = 0
             - E2/H2 · s2 + (E2/H2 + E3/H3) · s3  = 0
```

where s1, s2, s3 are the settlements at each layer interface.

In [ ]:
# ─────────────────────────────────────────────────────
# SOLVING LINEAR SYSTEMS — 3-layer settlement problem
# ─────────────────────────────────────────────────────

import numpy as np

# Soil layer properties
E = np.array([5000, 12000, 25000])  # Elastic modulus (kPa)
H = np.array([3.0, 4.0, 5.0])       # Layer thickness (m)
q = 150  # Applied foundation pressure (kPa)

# Stiffness per layer: k = E/H
k = E / H
print(f"Layer stiffness k = E/H: {np.round(k, 1)} kPa/m")
print()

# Build coefficient matrix A
A = np.array([
    [ k[0],       -k[0],          0     ],
    [-k[0],  k[0] + k[1],       -k[1]   ],
    [  0,        -k[1],    k[1] + k[2]   ],
])

# Right-hand side vector b
b = np.array([q, 0, 0])

print("Coefficient matrix A:")
print(np.round(A, 1))
print(f"\nLoad vector b: {b}")
print()

# Solve Ax = b
settlement = np.linalg.solve(A, b)

print("SOLUTION — Interface settlements:")
for i, s in enumerate(settlement):
    print(f"  s{i+1} = {s*1000:.2f} mm  (top of layer {i+1})")

# Layer compression = difference in settlement
compression = np.diff(np.append(settlement, 0))  # bottom is fixed
print(f"\nLayer compression:")
for i, c in enumerate(compression):
    print(f"  Layer {i+1}: {abs(c)*1000:.2f} mm")

total = settlement[0]
print(f"\nTotal surface settlement: {total*1000:.2f} mm")

# Verify: A @ x should equal b
residual = A @ settlement - b
print(f"Verification (A@x - b): {np.round(residual, 10)}  ← should be ~zero")

---

## Part 5: Eigenvalues & Eigenvectors — Principal Stresses

The **eigenvalue problem** for a stress tensor σ is:

```
σ · v = λ · v
```

- **Eigenvalues λ** = principal stresses (σ1, σ2, σ3)
- **Eigenvectors v** = principal directions (orientation of the planes where shear stress is zero)

This is the **exact same decomposition** used in PCA:
- In geotechnics: eigenvalues of stress tensor give principal stresses
- In ML: eigenvalues of covariance matrix give principal components

In [ ]:
# ─────────────────────────────────────────────────────
# EIGENVALUES & EIGENVECTORS — Principal stresses
# ─────────────────────────────────────────────────────

import numpy as np

# 2D stress tensor at a soil element
# σxx = 120 kPa (horizontal), σyy = 80 kPa (vertical), τxy = 25 kPa (shear)
sigma = np.array([
    [120, 25],
    [ 25, 80],
])

print("Stress tensor σ (kPa):")
print(sigma)
print()

# Compute eigenvalues and eigenvectors
eigenvalues, eigenvectors = np.linalg.eig(sigma)

# Sort by magnitude (σ1 > σ3)
idx = np.argsort(eigenvalues)[::-1]  # descending order
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print("PRINCIPAL STRESSES:")
print(f"  σ1 (major)  = {eigenvalues[0]:.2f} kPa")
print(f"  σ3 (minor)  = {eigenvalues[1]:.2f} kPa")
print()

# Principal directions
for i in range(2):
    angle = np.degrees(np.arctan2(eigenvectors[1, i], eigenvectors[0, i]))
    print(f"  Principal direction {i+1}: [{eigenvectors[0,i]:.4f}, {eigenvectors[1,i]:.4f}] → θ = {angle:.1f}°")
print()

# Derived quantities
sigma_mean = np.mean(eigenvalues)
tau_max = (eigenvalues[0] - eigenvalues[1]) / 2
deviatoric = eigenvalues[0] - eigenvalues[1]

print(f"Mean stress:            p = {sigma_mean:.2f} kPa")
print(f"Max shear stress:       τmax = {tau_max:.2f} kPa")
print(f"Deviatoric stress:      q = {deviatoric:.2f} kPa")
print()

# Verification: σ · v = λ · v
for i in range(2):
    lhs = sigma @ eigenvectors[:, i]
    rhs = eigenvalues[i] * eigenvectors[:, i]
    print(f"  Check σ·v{i+1} = λ{i+1}·v{i+1}: {np.allclose(lhs, rhs)}")

---

## Part 6: Mohr's Circle — Computed from Eigenvalues

**Mohr's circle** is the graphical representation of the 2D stress state. Once we have the principal stresses from eigenvalues, we can compute everything:

- **Center** = (σ1 + σ3) / 2
- **Radius** = (σ1 - σ3) / 2 = τmax
- **Any plane at angle θ:** σn = center + R cos(2θ), τ = R sin(2θ)

In [ ]:
# ─────────────────────────────────────────────────────
# MOHR'S CIRCLE — From eigenvalues of the stress tensor
# ─────────────────────────────────────────────────────

import numpy as np

# Stress tensor
sigma = np.array([
    [120, 25],
    [ 25, 80],
])

# Principal stresses via eigenvalues
eig_vals = np.linalg.eigvalsh(sigma)  # eigvalsh for symmetric matrices
sigma_1 = np.max(eig_vals)  # major principal stress
sigma_3 = np.min(eig_vals)  # minor principal stress

# Mohr's circle parameters
center = (sigma_1 + sigma_3) / 2
radius = (sigma_1 - sigma_3) / 2

print("MOHR'S CIRCLE PARAMETERS:")
print(f"  σ1 = {sigma_1:.2f} kPa  (major principal stress)")
print(f"  σ3 = {sigma_3:.2f} kPa  (minor principal stress)")
print(f"  Center = {center:.2f} kPa")
print(f"  Radius = τmax = {radius:.2f} kPa")
print()

# Stress on planes at various angles
print("STRESSES ON INCLINED PLANES:")
print(f"{'Angle θ':>8} {'σn (kPa)':>10} {'τ (kPa)':>10}")
print("─" * 30)
for theta_deg in [0, 15, 30, 45, 60, 75, 90]:
    theta_rad = np.radians(theta_deg)
    sigma_n = center + radius * np.cos(2 * theta_rad)
    tau = radius * np.sin(2 * theta_rad)
    print(f"{theta_deg:6d}°  {sigma_n:10.2f}  {tau:10.2f}")

print()
print(f"At θ=0°:  σn = σ1 = {sigma_1:.2f}, τ = 0  (principal plane)")
print(f"At θ=45°: σn = {center:.2f}, τ = τmax = {radius:.2f}  (max shear plane)")
print(f"At θ=90°: σn = σ3 = {sigma_3:.2f}, τ = 0  (principal plane)")

---

## Part 7: SVD Basics — Dimensionality Reduction Preview

**Singular Value Decomposition (SVD)** factorizes any matrix into three components:

```
A = U × S × V^T
```

- **U** = left singular vectors (patterns in rows / samples)
- **S** = singular values (importance of each pattern)
- **V^T** = right singular vectors (patterns in columns / features)

In ML, SVD is used for:
- **PCA** (keeping top k singular values = top k principal components)
- **Noise reduction** (small singular values often represent noise)
- **Data compression** (approximate a matrix with fewer components)

Let us apply it to a soil property dataset.

In [ ]:
# ─────────────────────────────────────────────────────
# SVD — Dimensionality reduction with soil property data
# ─────────────────────────────────────────────────────

import numpy as np

# Soil property data: 8 samples × 5 properties
# Columns: [N60, Vs(m/s), γ(kN/m3), e(void ratio), PI(%)]
soil_data = np.array([
    [ 5, 140, 16.5, 0.85, 25],
    [ 8, 165, 17.0, 0.78, 22],
    [12, 190, 17.8, 0.70, 18],
    [18, 220, 18.5, 0.62, 12],
    [28, 260, 19.2, 0.55,  8],
    [35, 290, 19.8, 0.48,  5],
    [50, 330, 20.5, 0.40,  3],
    [60, 360, 21.0, 0.35,  2],
], dtype=float)

properties = ["N60", "Vs", "gamma", "void_ratio", "PI"]
print(f"Soil data shape: {soil_data.shape} (8 samples × 5 properties)")
print()

# Center the data (subtract column means)
means = np.mean(soil_data, axis=0)
centered = soil_data - means

# SVD decomposition
U, S, Vt = np.linalg.svd(centered, full_matrices=False)

print(f"U shape:  {U.shape}   (8 samples × 5 components)")
print(f"S values: {np.round(S, 2)}")
print(f"Vt shape: {Vt.shape}  (5 components × 5 properties)")
print()

# How much variance does each component explain?
variance_explained = S**2 / np.sum(S**2) * 100
cumulative = np.cumsum(variance_explained)

print("VARIANCE EXPLAINED:")
print(f"{'Component':>10} {'Singular Val':>12} {'Variance %':>10} {'Cumulative %':>12}")
print("─" * 48)
for i in range(len(S)):
    print(f"  PC-{i+1:d}      {S[i]:12.2f}  {variance_explained[i]:9.1f}%  {cumulative[i]:11.1f}%")
print()
print(f"First 2 components capture {cumulative[1]:.1f}% of all variance!")
print("This means 5 soil properties can be summarized by ~2 numbers.")

---

## Part 8: Determinant & Inverse — Checking Stiffness Matrices

| Function | Purpose | Geotechnical Use |
|----------|---------|------------------|
| `np.linalg.det(A)` | Determinant | Is the stiffness matrix singular (det = 0)? |
| `np.linalg.inv(A)` | Inverse | Compliance matrix = inverse of stiffness |
| `np.linalg.cond(A)` | Condition number | Is the system ill-conditioned (numerically unstable)? |

A **singular** matrix (det = 0) means the system has no unique solution — this happens when soil layers have identical properties or the model has redundant constraints.

In [ ]:
# ─────────────────────────────────────────────────────
# DETERMINANT & INVERSE — Stiffness matrix checks
# ─────────────────────────────────────────────────────

import numpy as np

# Stiffness matrix for a simple 2D foundation model
K = np.array([
    [2000, -800,    0],
    [-800, 2400, -600],
    [   0, -600, 1800],
], dtype=float)

print("Stiffness matrix K (kN/m):")
print(K)
print()

# Determinant — if zero, matrix is singular (no unique solution)
det_K = np.linalg.det(K)
print(f"Determinant: {det_K:,.0f}")
print(f"Singular?    {'YES — problem!' if abs(det_K) < 1e-10 else 'No — system is solvable'}")
print()

# Condition number — large values mean numerical instability
cond_K = np.linalg.cond(K)
print(f"Condition number: {cond_K:.1f}")
if cond_K > 1e10:
    print("  WARNING: Ill-conditioned — solution may be unreliable!")
else:
    print("  Well-conditioned — solution is reliable.")
print()

# Inverse = compliance (flexibility) matrix
C = np.linalg.inv(K)
print("Compliance matrix C = K^{-1} (m/kN):")
print(np.round(C, 8))
print()

# Verify: K @ C should be identity
identity = K @ C
print(f"K @ K^(-1) = I?  {np.allclose(identity, np.eye(3))}")
print()

# Use: displacement = C @ force
F = np.array([150, 0, 0])  # 150 kPa applied at node 1
disp = C @ F
print(f"Force vector:        {F} kN")
print(f"Displacement (m):    {np.round(disp, 6)}")
print(f"Displacement (mm):   {np.round(disp * 1000, 3)}")

---

## Part 9: Complete Application — Principal Stresses & Mohr's Circle

A soil element at 8 m depth below a loaded embankment is subjected to:
- Vertical stress σyy = 180 kPa
- Horizontal stress σxx = 95 kPa
- Shear stress τxy = 40 kPa

**Compute:**
1. Principal stresses and directions (eigenvalue decomposition)
2. Mohr's circle parameters
3. Stresses on planes at 15° intervals
4. Check against Mohr-Coulomb failure: τf = c' + σn × tan(φ')

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE APPLICATION: PRINCIPAL STRESS ANALYSIS
# ══════════════════════════════════════════════════════

import numpy as np

# ─── Problem definition ───
sigma_xx = 95.0    # horizontal stress (kPa)
sigma_yy = 180.0   # vertical stress (kPa)
tau_xy = 40.0      # shear stress (kPa)

# Soil strength parameters
c_prime = 10.0     # effective cohesion (kPa)
phi_prime = 30.0   # effective friction angle (degrees)

# Build the stress tensor
sigma = np.array([
    [sigma_xx, tau_xy],
    [tau_xy,   sigma_yy],
])

print("═" * 60)
print("  PRINCIPAL STRESS ANALYSIS")
print("  Soil element at 8 m depth under embankment")
print("═" * 60)
print(f"\n  Stress tensor (kPa):")
print(f"    | {sigma_xx:7.1f}  {tau_xy:7.1f} |    | σxx  τxy |")
print(f"    | {tau_xy:7.1f}  {sigma_yy:7.1f} |    | τxy  σyy |")
print()

# ─── Step 1: Eigenvalue decomposition ───
eigenvalues, eigenvectors = np.linalg.eigh(sigma)  # eigh for symmetric

# Sort descending (σ1 > σ3)
idx = np.argsort(eigenvalues)[::-1]
sigma_1 = eigenvalues[idx[0]]
sigma_3 = eigenvalues[idx[1]]
v1 = eigenvectors[:, idx[0]]
v3 = eigenvectors[:, idx[1]]

theta_p = np.degrees(np.arctan2(v1[1], v1[0]))

print("STEP 1: PRINCIPAL STRESSES (from eigenvalues)")
print(f"  σ1 = {sigma_1:.2f} kPa  (major)")
print(f"  σ3 = {sigma_3:.2f} kPa  (minor)")
print(f"  Direction of σ1: θ = {theta_p:.1f}° from x-axis")
print()

# ─── Step 2: Mohr's circle parameters ───
center = (sigma_1 + sigma_3) / 2
radius = (sigma_1 - sigma_3) / 2

print("STEP 2: MOHR'S CIRCLE")
print(f"  Center (mean stress p):  {center:.2f} kPa")
print(f"  Radius (max shear τmax): {radius:.2f} kPa")
print(f"  Deviatoric stress q:     {2*radius:.2f} kPa")
print()

In [ ]:
# ─────────────────────────────────────────────────────
# COMPLETE APPLICATION (cont.) — Plane stresses & failure check
# ─────────────────────────────────────────────────────

# ─── Step 3: Stresses on planes at 15° intervals ───
print("STEP 3: STRESSES ON INCLINED PLANES")
print(f"{'Plane θ':>8} {'σn (kPa)':>10} {'τ (kPa)':>10} {'τf (kPa)':>10} {'FoS':>6}")
print("─" * 48)

phi_rad = np.radians(phi_prime)
critical_planes = []

for theta_deg in range(0, 91, 15):
    theta_rad = np.radians(theta_deg)
    sigma_n = center + radius * np.cos(2 * theta_rad)
    tau = abs(radius * np.sin(2 * theta_rad))
    
    # Mohr-Coulomb failure shear stress
    tau_f = c_prime + sigma_n * np.tan(phi_rad)
    
    # Factor of safety
    fos = tau_f / tau if tau > 0 else float('inf')
    
    marker = " ← CRITICAL" if fos < 1.5 and tau > 0 else ""
    fos_str = f"{fos:.2f}" if fos < 100 else "∞"
    print(f"{theta_deg:6d}°  {sigma_n:10.2f}  {tau:10.2f}  {tau_f:10.2f}  {fos_str:>6}{marker}")
    
    if tau > 0:
        critical_planes.append((theta_deg, fos))

print()

# ─── Step 4: Most critical plane ───
if critical_planes:
    min_fos_plane = min(critical_planes, key=lambda x: x[1])
    print(f"STEP 4: FAILURE ASSESSMENT")
    print(f"  Soil: c' = {c_prime} kPa, φ' = {phi_prime}°")
    print(f"  Most critical plane: θ = {min_fos_plane[0]}°, FoS = {min_fos_plane[1]:.2f}")
    if min_fos_plane[1] >= 1.5:
        print(f"  Status: SAFE (FoS >= 1.5)")
    elif min_fos_plane[1] >= 1.0:
        print(f"  Status: MARGINAL (1.0 <= FoS < 1.5)")
    else:
        print(f"  Status: FAILURE (FoS < 1.0)")

# ─── Analytical critical angle ───
theta_crit = 45 + phi_prime / 2
print(f"\n  Theoretical failure plane: θ = 45 + φ'/2 = {theta_crit:.0f}°")
print("═" * 48)

---

## Practice Exercises

### Exercise 1: 3D Principal Stresses
A soil element has σxx=100, σyy=200, σzz=150, τxy=30, τxz=10, τyz=20 (all kPa). Build the 3×3 stress tensor and use `np.linalg.eigh()` to find the three principal stresses.

### Exercise 2: Solve a 4-Layer Settlement
Extend Part 4 to 4 layers with E = [3000, 8000, 15000, 30000] kPa and H = [2.0, 3.0, 4.0, 5.0] m. Solve for the settlements under q = 200 kPa.

### Exercise 3: SVD on Borehole Data
Create a 10×4 array of soil data (columns: N60, Vs, density, water content). Apply SVD. How many components are needed to capture 95% of the variance?

In [ ]:
# EXERCISE 1
# sigma_3d = np.array([
#     [100, 30, 10],
#     [ 30, 200, 20],
#     [ 10, 20, 150],
# ])
# eigenvalues, eigenvectors = np.linalg.eigh(sigma_3d)
# print(f"σ1 = {eigenvalues[2]:.2f}, σ2 = {eigenvalues[1]:.2f}, σ3 = {eigenvalues[0]:.2f}")

In [ ]:
# EXERCISE 2
# E = np.array([3000, 8000, 15000, 30000])
# H = np.array([2.0, 3.0, 4.0, 5.0])
# k = E / H
# Build 4×4 tridiagonal matrix A, solve A @ s = b

In [ ]:
# EXERCISE 3
# np.random.seed(42)
# soil_data = ...
# centered = soil_data - np.mean(soil_data, axis=0)
# U, S, Vt = np.linalg.svd(centered, full_matrices=False)
# variance = S**2 / np.sum(S**2) * 100
# cumulative = np.cumsum(variance)
# Find where cumulative >= 95

---

## Key Takeaways

1. **Stress tensors are matrices**, and finding principal stresses is an **eigenvalue problem** — the same math behind PCA in machine learning.

2. **`np.linalg.solve(A, b)`** solves systems of equations without computing the inverse — faster and more numerically stable than A^{-1} @ b.

3. **Eigenvalues** of a stress tensor give principal stresses; **eigenvectors** give principal directions. For symmetric matrices, use `np.linalg.eigh()` (faster, guaranteed real results).

4. **SVD** decomposes data into ranked components by importance. In our soil data, 2 components captured most of the variance — meaning 5 properties are largely redundant.

5. **Determinant and condition number** tell you whether a stiffness matrix is solvable and numerically reliable — essential checks before trusting any FEM result.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| Eigenvalues of stress tensor | Eigenvalues of covariance matrix (PCA) |
| Principal stresses (σ1, σ3) | Principal components (PC1, PC2) |
| SVD of soil data matrix | SVD for dimensionality reduction, recommender systems |
| Solving Ax = b | Normal equations in linear regression |
| Condition number | Detecting multicollinearity in features |
| Matrix inverse | Closed-form solution: β = (X^T X)^{-1} X^T y |
| Symmetric matrices (stress tensor) | Covariance matrices are always symmetric |
| Variance explained by eigenvalues | Scree plot, choosing number of components |

The linear algebra you use for Mohr's circle today is the **exact same math** that powers PCA, SVD, and linear regression in machine learning.

---

*Day 017 of 100 — Principal stresses and principal components share the same eigenvalue decomposition.*